In [ ]:
import numpy as np
import random
import torch

from matplotlib import pyplot as plt
from torch.utils.data import DataLoader

from datasets.hyspecnet11k import HySpecNet11k
from metrics import psnr, sa

from models import models
from utils import checkpoint

torch.manual_seed(10587)
random.seed(10857)

In [ ]:
model_dict = {
    # # 1D-CAE
    "cae1d_cr004": "./results/trains/current/hyspecnet11k/_cae1d/cae1d_cr004-mse-easy-0.0001/final.pth.tar",
    "cae1d_cr008": "./results/trains/current/hyspecnet11k/_cae1d/cae1d_cr008-mse-easy-0.0001/final.pth.tar",
    "cae1d_cr016": "./results/trains/current/hyspecnet11k/_cae1d/cae1d_cr016-mse-easy-0.0001/final.pth.tar",
    "cae1d_cr032": "./results/trains/current/hyspecnet11k/_cae1d/cae1d_cr032-mse-easy-0.0001/final.pth.tar",

    # # SSCNet
    "sscnet_cr004": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr004-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr008": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr008-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr016": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr016-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr032": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr032-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr051": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr051-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr101": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr101-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr202": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr202-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr269": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr269-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr512": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr512-mse-easy-1e-05/final.pth.tar",
    "sscnet_cr1024": "./results/trains/current/hyspecnet11k/_sscnet/sscnet_cr1024-mse-easy-1e-05/final.pth.tar",

    # # 3D-CAE
    "cae3d_cr004": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr004-mse-easy-0.0001/final.pth.tar",
    "cae3d_cr008": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr008-mse-easy-0.0001/final.pth.tar",
    "cae3d_cr016": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr016-mse-easy-0.0001/final.pth.tar",
    "cae3d_cr032": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr032-mse-easy-0.0001/final.pth.tar",
    "cae3d_cr051": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr051-x-mse-easy-1e-05/final.pth.tar",
    "cae3d_cr127": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr127-x-mse-easy-1e-05/final.pth.tar",
    "cae3d_cr253": "./results/trains/current/hyspecnet11k/_cae3d/cae3d_cr253-x-mse-easy-1e-05/final.pth.tar",

    # # HyCoT
    "hycot_cr004": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr004_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr008": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr008_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr016": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr016_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr032": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr032_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr064": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr064_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr128": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr128_pd4-easy_random_16x16-mse-0.001/final.pth.tar",
    "hycot_cr256": "./results/trains/current/hyspecnet11k/_hycot/hycot_cr256_pd4-easy_random_16x16-mse-0.001/final.pth.tar",

    # OURS
    "hycass_cr004_spatial0x_n1024": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr004_spatial0x_n1024-random_16x16-mse-0.001/final.pth.tar",
    "hycass_cr004_spatial3x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr004_spatial3x_n128-x-mse-0.0001/final.pth.tar",
    "hycass_cr008_spatial0x_n1024": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr008_spatial0x_n1024-random_16x16-mse-0.001/final.pth.tar",
    "hycass_cr016_spatial0x_n1024": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr016_spatial0x_n1024-random_16x16-mse-0.001/final.pth.tar",
    "hycass_cr032_spatial0x_n1024": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr032_spatial0x_n1024-random_16x16-mse-0.001/final.pth.tar",
    "hycass_cr101_spatial1x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr101_spatial1x_n128-x-mse-0.0001/final.pth.tar",
    "hycass_cr101_spatial2x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr101_spatial2x_n128-x-mse-0.0001/final.pth.tar",
    "hycass_cr101_spatial3x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr101_spatial3x_n128-x-mse-0.0001/final.pth.tar",
    "hycass_cr202_spatial2x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr202_spatial2x_n128-x-mse-0.0001/final.pth.tar",
    "hycass_cr404_spatial3x_n128": "./results/trains/current/hyspecnet11k/_ticmod/easy-hycass_cr404_spatial3x_n128-x-mse-0.0001/final.pth.tar",
}

num_models = len(model_dict)

path_save = "./results/visualizations/"

path_dataset = "./datasets/hyspecnet-11k/"
mode = "easy"
rgb_bands = [44, 29, 11]

idx_hsi_pick = [11]

num_visualizations = 4
sa_min = 0
sa_max = 10
psnr_min = 30
psnr_max = 60

In [ ]:
def save_image(data, filename, cmap=None, vmin=None, vmax=None):
    sizes = np.shape(data)
    fig = plt.figure()
    fig.set_size_inches(1. * sizes[0] / sizes[1], 1, forward = False)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    plt.savefig(filename, dpi = sizes[0]) 
    plt.close()

In [ ]:
test_dataset = HySpecNet11k(path_dataset, mode=mode, split="test", transform=None)
test_dataloader = DataLoader(
        test_dataset,
        batch_size=1,
        num_workers=4,
        shuffle=True,
        pin_memory=False,
        drop_last=False
    )

psnr_metric = psnr.PeakSignalToNoiseRatio()
sa_metric = sa.SpectralAngle()

In [ ]:
for n in idx_hsi_pick:
    print(test_dataset.npy_paths[n])

    data_org = test_dataset[n].unsqueeze(0)
    data_org_plt = data_org.squeeze(0).detach().cpu().permute(1, 2, 0)

    # RGB ORIGINAL
    fig = plt.figure(figsize=(40, len(model_dict)*10))
    fig.add_subplot(len(model_dict)+1, num_visualizations, 1)
    data_org_plt_rgb = data_org_plt[:, :, rgb_bands]
    plt.imshow(data_org_plt_rgb)
    plt.axis("off")
    plt.title("Original RGB \n \n")
    save_image(data_org_plt_rgb, f"{path_save}/org_rgb.png")

    for i, (model_name, model_weights) in enumerate(model_dict.items()):
        # MODEL LOAD
        model = models[model_name]().eval()
        checkpoint.load_checkpoint_eval(model_weights, model)

        # COMPRESS AND RECONSTRUCT
        data_rec = model(data_org)

        # METRIC CALCULATION
        psnr = psnr_metric(data_org, data_rec)
        sa = sa_metric(data_org, data_rec)

        # RGB RECONSTRUCTED
        data_rec_plt = data_rec.squeeze(0).detach().cpu().permute(1, 2, 0)
        data_rec_plt_rgb = data_rec_plt[:, :, rgb_bands]

        fig.add_subplot(num_models+1, num_visualizations, i*num_visualizations+2)
        plt.imshow(data_rec_plt_rgb)
        plt.axis("off")
        plt.title(f"Reconstructed RGB \n {model_name} \n {psnr:.2f} dB")
        save_image(data_rec_plt_rgb, f"{path_save}/rec_rgb-{model_name}-{psnr:.2f}dB-{model.compression_ratio:.2f}CR.png")

        # SA PIXELWISE
        map_sa = torch.rad2deg(torch.acos(torch.sum(data_org_plt * data_rec_plt, dim=2) / torch.sqrt(torch.sum(data_org_plt ** 2, dim=2) * torch.sum(data_rec_plt ** 2, dim=2))))

        fig.add_subplot(num_models+1, num_visualizations, i*num_visualizations+3)
        plt.imshow(map_sa, cmap="jet", vmin=sa_min, vmax=sa_max)
        plt.colorbar()
        plt.axis("off")
        plt.title(f"Pixelwise SA \n {model_name} \n")
        save_image(map_sa, f"{path_save}/sa_map-{model_name}.png", cmap="jet", vmin=sa_min, vmax=sa_max)

        # SPECTRUMS ORIGINAL & RECONSTRUCTED
        tensor_size = data_org.size()
        index = tensor_size[2] // 2

        (idx_min_x, idx_min_y) = tuple((map_sa==torch.min(map_sa)).nonzero().tolist()[0])
        (idx_max_x, idx_max_y) = tuple((map_sa==torch.max(map_sa)).nonzero().tolist()[0])

        data_org_spec_min = data_org.squeeze(0).detach().cpu()[:, idx_min_x, idx_min_y]
        data_rec_spec_min = data_rec.squeeze(0).detach().cpu()[:, idx_min_x, idx_min_y]

        data_org_spec_max = data_org.squeeze(0).detach().cpu()[:, idx_max_x, idx_max_y]
        data_rec_spec_max = data_rec.squeeze(0).detach().cpu()[:, idx_max_x, idx_max_y]

        fig.add_subplot(num_models+1, num_visualizations, i*num_visualizations+4)
        num_bands = data_org.shape[1]
        bands = [i for i in range(1, num_bands+1)]
        
        plt.plot(bands, data_org_spec_min)
        plt.plot(bands, data_rec_spec_min)

        plt.plot(bands, data_org_spec_max)
        plt.plot(bands, data_rec_spec_max)

        plt.xlim(1, num_bands)
        plt.ylim(0, 1)
        plt.grid(which='both')
        plt.xlabel('Band')
        plt.ylabel('Value')
        plt.legend(['org-best', 'rec-best', 'org-worst', 'rec-worst'], loc='upper right')
        plt.title(f"Spectrums \n {model_name} \n")

    plt.show()